# Your Title Here

**Name(s)**: (your name(s) here)

**Website Link**: (your website link)

In [21]:
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
pd.options.plotting.backend = 'plotly'

import plotly.graph_objects as go

from numpy.random import default_rng
rng = default_rng(42)

DATA_PATH = Path("data/2025_LoL_esports_match_data_from_OraclesElixir.csv")


from dsc80_utils import *

## Step 1: Introduction

In [22]:
df = pd.read_csv(DATA_PATH)
df.head()

C:\Users\Alexa\AppData\Local\Temp\ipykernel_21356\2006162025.py:1: DtypeWarning:

Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.



,gameid,datacompleteness,url,league,...,deathsat25,opp_killsat25,opp_assistsat25,opp_deathsat25
0,LOLTMNT03_179647,complete,NaN,LFL2,...,2.0,2.0,4.0,2.0
1,LOLTMNT03_179647,complete,NaN,LFL2,...,2.0,1.0,7.0,0.0
2,LOLTMNT03_179647,complete,NaN,LFL2,...,2.0,1.0,5.0,1.0
3,LOLTMNT03_179647,complete,NaN,LFL2,...,2.0,6.0,2.0,0.0
4,LOLTMNT03_179647,complete,NaN,LFL2,...,2.0,0.0,8.0,0.0


In [23]:
df.shape

(120636, 165)

In [24]:
overview = pd.DataFrame({
    "column": df.columns,
    "dtype": [df[c].dtype for c in df.columns],
    "missing_rate": df.isna().mean().values
}).sort_values("missing_rate", ascending=False)

overview.head(20)

,column,dtype,missing_rate
57,dragons (type unknown),float64,0.99
102,monsterkillsownjungle,float64,0.92
103,monsterkillsenemyjungle,float64,0.92
...,...,...,...
76,opp_turretplates,float64,0.85
98,gpr,float64,0.85
74,firsttothreetowers,float64,0.85


In [25]:
PROJECT_QUESTION = "Can early-game (15-minute) statistics predict who wins a professional League of Legends match?"

relevant_cols = [
    "gameid", "league", "year", "split", "playoffs", "patch", "date",
    "side", "position", "datacompleteness",
    "result", "firstblood", "firstdragon", "firsttower",
    "golddiffat15", "xpdiffat15", "csdiffat15",
    "golddiffat10", "xpdiffat10", "csdiffat10"
]
relevant_cols = [c for c in relevant_cols if c in df.columns]

print(PROJECT_QUESTION)
print("Rows:", df.shape[0])
print("Relevant columns present:", relevant_cols)

Can early-game (15-minute) statistics predict who wins a professional League of Legends match?
Rows: 120636
Relevant columns present: ['gameid', 'league', 'year', 'split', 'playoffs', 'patch', 'date', 'side', 'position', 'datacompleteness', 'result', 'firstblood', 'firstdragon', 'firsttower', 'golddiffat15', 'xpdiffat15', 'csdiffat15', 'golddiffat10', 'xpdiffat10', 'csdiffat10']


## Step 2: Data Cleaning and Exploratory Data Analysis

In [26]:
team = df.copy()

# Team-level rows
team = team[team["position"].astype(str).str.lower() == "team"].copy()

# Complete data only (if available)
if "datacompleteness" in team.columns:
    team = team[team["datacompleteness"].astype(str).str.lower() == "complete"].copy()

# Standardize side formatting
team["side"] = team["side"].astype(str).str.title()

# Coerce key columns to numeric
num_cols = [
    "result","firstblood","firstdragon","firsttower",
    "golddiffat15","xpdiffat15","csdiffat15",
    "golddiffat10","xpdiffat10","csdiffat10",
]
for c in num_cols:
    if c in team.columns:
        team[c] = pd.to_numeric(team[c], errors="coerce")

team["win"] = team["result"]

print("Team rows:", team.shape[0])
print("Unique games:", team["gameid"].nunique())
team[["gameid","side","win"]].head()

Team rows: 18472
Unique games: 9236


,gameid,side,win
10,LOLTMNT03_179647,Blue,0
11,LOLTMNT03_179647,Red,1
22,LOLTMNT06_96134,Blue,1
23,LOLTMNT06_96134,Red,0
34,LOLTMNT06_95160,Blue,0


In [27]:
clean_head_cols = [c for c in [
    "gameid","league","patch","side","win",
    "firstblood","firstdragon","firsttower",
    "golddiffat15","xpdiffat15","csdiffat15"
] if c in team.columns]

team[clean_head_cols].head()

,gameid,league,patch,side,...,firsttower,golddiffat15,xpdiffat15,csdiffat15
10,LOLTMNT03_179647,LFL2,15.01,Blue,...,0.0,-3837.0,-469.0,-16.0
11,LOLTMNT03_179647,LFL2,15.01,Red,...,1.0,3837.0,469.0,16.0
22,LOLTMNT06_96134,LFL2,15.01,Blue,...,1.0,5069.0,2014.0,64.0
23,LOLTMNT06_96134,LFL2,15.01,Red,...,0.0,-5069.0,-2014.0,-64.0
34,LOLTMNT06_95160,LFL2,15.01,Blue,...,0.0,118.0,1990.0,-43.0


In [28]:
# Plot 1: distribution of golddiffat15
fig_u1 = px.histogram(team.dropna(subset=["golddiffat15"]),
                      x="golddiffat15",
                      nbins=60,
                      title="Distribution of Gold Difference at 15 Minutes (Team-Level)")
fig_u1.update_layout(xaxis_title="Gold Diff at 15", yaxis_title="Count")
fig_u1.show()

In [29]:
# Plot 2: win rate by side
win_by_side = team.groupby("side")["win"].mean().reset_index()
fig_u2 = px.bar(win_by_side, x="side", y="win",
                title="Win Rate by Side",
                labels={"win":"Win Rate"})
fig_u2.update_layout(yaxis_tickformat=".0%")
fig_u2.show()

In [30]:
fig = px.scatter(
    team,
    x="golddiffat15",
    y="xpdiffat15",
    color="win",
    opacity=0.5,
    color_continuous_scale=["purple", "orange"],
    title="Gold Difference vs XP Difference at 15 Minutes"
)

fig.update_layout(
    xaxis_title="Gold Difference at 15",
    yaxis_title="XP Difference at 15",
    coloraxis_colorbar_title="Win"
)

fig.show()

In [31]:
# Estimate win probability by smoothing
team_sorted = team.dropna(subset=["golddiffat15"]).sort_values("golddiffat15")

rolling = (
    team_sorted
    .rolling(window=300, on="golddiffat15")["win"]
    .mean()
)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=team_sorted["golddiffat15"],
    y=rolling,
    mode="lines"
))

fig.update_layout(
    title="Smoothed Win Probability vs Gold Difference at 15",
    yaxis_tickformat=".0%"
)

fig.show()

In [32]:
agg = team.groupby("firstblood")[["win","golddiffat15","xpdiffat15","csdiffat15"]].mean().reset_index()
agg

,firstblood,win,golddiffat15,xpdiffat15,csdiffat15
0,0.0,0.42,-762.51,-347.93,-5.17
1,1.0,0.58,762.51,347.93,5.17


## Step 3: Assessment of Missingness

In [33]:
team["gold25_missing"] = team["golddiffat25"].isna()

In [34]:
team["gold25_missing"].mean()

0.03724556084885232

In [40]:
fig_missing = px.histogram(
    team,
    x="gamelength",
    color="gold25_missing",
    title="Game Length Distribution by Missingness of golddiffat25",
    labels={"gold25_missing":"golddiffat25 Missing"}
)

fig_missing.show()

In [41]:
observed_stat = (
    team.groupby("gold25_missing")["gamelength"].mean().diff().iloc[-1]
)

observed_stat

-604.122797863771

In [42]:
n_perm = 3000
perm_stats = []

for i in range(n_perm):

    shuffled = team.copy()
    shuffled["gold25_missing"] = rng.permutation(shuffled["gold25_missing"])

    stat = (
        shuffled.groupby("gold25_missing")["gamelength"].mean().diff().iloc[-1]
    )

    perm_stats.append(stat)

perm_stats = np.array(perm_stats)

In [43]:
p_value = np.mean(np.abs(perm_stats) >= np.abs(observed_stat))

observed_stat, p_value

(-604.122797863771, 0.0)

In [44]:
fig_perm_missing = px.histogram(
    x=perm_stats,
    nbins=60,
    title="Permutation Distribution for Missingness Dependency Test"
)

fig_perm_missing.add_vline(
    x=observed_stat,
    line_width=3,
    annotation_text="Observed Statistic"
)

fig_perm_missing.update_layout(
    xaxis_title="Difference in Mean Game Length",
    yaxis_title="Count"
)

fig_perm_missing.show()

In [45]:
observed_side = (
    team.groupby("gold25_missing")["side"]
    .apply(lambda x: (x=="Blue").mean())
    .diff().iloc[-1]
)

### MNAR Analysis

One column that may be Missing Not At Random (MNAR) is `golddiffat25`.

In professional League of Legends matches, statistics recorded at the 25-minute mark may be missing when games end before that time. In this case, the missingness depends on the duration of the match itself. Because shorter games end before reaching 25 minutes, statistics at that time point cannot be recorded.

Therefore, the missingness depends on the underlying data generating process (game duration), making it plausible that this variable is MNAR.

If we had additional data explicitly indicating whether a match reached 25 minutes, the missingness could potentially be modeled as MAR instead

## Step 4: Hypothesis Testing

In [46]:
# Create indicator: positive gold diff at 15
team["ahead15"] = (team["golddiffat15"] > 0).astype(int)

team[["golddiffat15","ahead15","win"]].head()

,golddiffat15,ahead15,win
10,-3837.0,0,0
11,3837.0,1,1
22,5069.0,1,1
23,-5069.0,0,0
34,118.0,1,0


In [47]:
obs_stat = (
    team.groupby("ahead15")["win"].mean().loc[1]
    - team.groupby("ahead15")["win"].mean().loc[0]
)

obs_stat

0.4576656614933269

In [48]:
n_perm = 3000
perm_stats = []

for i in range(n_perm):

    shuffled = team.copy()
    shuffled["ahead15"] = rng.permutation(shuffled["ahead15"])

    stat = (
        shuffled.groupby("ahead15")["win"].mean().loc[1]
        - shuffled.groupby("ahead15")["win"].mean().loc[0]
    )

    perm_stats.append(stat)

perm_stats = np.array(perm_stats)

In [49]:
p_value = np.mean(perm_stats >= obs_stat)

obs_stat, p_value

(0.4576656614933269, 0.0)

In [50]:
fig_perm = px.histogram(
    x=perm_stats,
    nbins=60,
    title="Permutation Distribution of Win Rate Difference"
)

fig_perm.add_vline(
    x=obs_stat,
    line_width=3,
    annotation_text=f"Observed={obs_stat:.3f}"
)

fig_perm.update_layout(
    xaxis_title="WinRate(ahead) - WinRate(behind)",
    yaxis_title="Count"
)

fig_perm.show()

### Hypothesis Test

Null Hypothesis:  
Teams with a gold advantage at 15 minutes win at the same rate as teams without a gold advantage.

Alternative Hypothesis:  
Teams with a gold advantage at 15 minutes win at a higher rate than teams without a gold advantage.

Test Statistic:  
Difference in win rate between teams ahead in gold at 15 minutes and teams behind.

We conduct a permutation test with 3000 permutations to evaluate whether the observed difference in win rates could occur by chance.

## Step 5: Framing a Prediction Problem

In [ ]:
# TODO

### Prediction Problem

We aim to predict whether a team will win a professional League of Legends match using statistics available at the 15-minute mark.

This is a **binary classification problem** because the response variable (`win`) takes two values: 1 (win) and 0 (loss).

Response Variable:
`win`

Features available at time of prediction:
- `golddiffat15`
- `xpdiffat15`
- `csdiffat15`
- `firstblood`
- `firstdragon`
- `firsttower`

These features are available early in the match and do not rely on information from later stages of the game.

Evaluation Metric:
We use **accuracy** and **F1 score** because the target variable is binary and these metrics measure how well the model correctly predicts match outcomes.

## Step 6: Baseline Model

In [51]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

In [52]:
features = [
    "golddiffat15",
    "xpdiffat15",
    "csdiffat15",
    "firstblood"
]

baseline_df = team[features + ["win"]].dropna()

In [53]:
X = baseline_df[features]
y = baseline_df["win"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

In [54]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), features)
    ]
)

baseline_model = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression())
])

In [55]:
baseline_model.fit(X_train, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['golddiffat15', 'xpdiffat15',
                                                   'csdiffat15',
                                                   'firstblood'])])),
                ('model', LogisticRegression())])

In [56]:
preds = baseline_model.predict(X_test)

baseline_accuracy = accuracy_score(y_test, preds)
baseline_f1 = f1_score(y_test, preds)

baseline_accuracy, baseline_f1

(0.7379818103074924, 0.737527114967462)

### Baseline Model

Our baseline model is a **logistic regression classifier** predicting match outcome (`win`).

Features used:
- `golddiffat15` (quantitative)
- `xpdiffat15` (quantitative)
- `csdiffat15` (quantitative)
- `firstblood` (binary)

These features capture early-game advantages in gold, experience, and minion farming.

All preprocessing and model training steps are implemented inside a **single sklearn Pipeline**.

Model performance is evaluated using **accuracy** and **F1 score** on a held-out test dataset

## Step 7: Final Model

In [38]:
# TODO

## Step 8: Fairness Analysis

In [39]:
# TODO